# 17 — Emergent factor discovery (5,000-user cohort)

Interpret the already-trained **Real RBM** (`channelB1`, no personality signal ever shown to it) post-hoc, instead of assuming what "personality" is.

- **Movie side**: for each of the 128 hidden units, find top-weighted movies and explain them via genres, release year, MovieLens genome tags, and user free-text tags.
- **User side**: correlate hidden user activations (`H1`) against derived behavioral features (leniency, extremity, activity, genre entropy, contrarian score, popularity bias) — kept separate, not pre-merged, since they are statistically independent axes.
- **Ablation**: test whether the existing hyperbolic-joint architecture earns its keep over a naive learned per-user bias.

All analysis runs on **train-partition data only** (`mask==1`), consistent with how `H1`, `personality.npy`, and `user_means.npy` were derived in 09b/12b/14b.

| Part | Section |
|------|---------|
| 0 | Setup |
| 1 | Movie-side metadata assembly |
| 2 | Movie-side hidden-unit interpretation |
| 3 | User-side behavioral feature table |
| 4 | Correlation analysis (user-side) |
| 5 | Synthesis |
| 6 | Ablation: hyperbolic joint vs. naive learned bias |
| 7 | Documentation |

## Part 0 — Setup

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent

data_dir = root / "data"
proc = root / "data" / "processed"
out_dir = root / "outputs"
out_dir.mkdir(parents=True, exist_ok=True)

RATING_LEVELS = np.array([0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0], dtype=np.float64)
K = len(RATING_LEVELS)


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -60.0, 60.0)))


paths = {
    "W1": proc / "rbmB1_weights_5k.npy",
    "bh1": proc / "rbmB1_bias_hidden_5k.npy",
    "channelB1": proc / "channelB1_softmax.npy",
    "mask": proc / "mask.npy",
    "cohort": proc / "cohort_user_ids.npy",
    "vocab": proc / "movie_vocab.npy",
    "means": proc / "user_means.npy",
    "personality": proc / "personality.npy",
    "mu_global": proc / "mu_global.npy",
    "test": proc / "test_labels.csv",
}
for p in paths.values():
    assert p.exists(), f"Missing {p}"

W1 = np.load(paths["W1"])
bh1 = np.load(paths["bh1"])
channelB1 = np.load(paths["channelB1"], mmap_mode="r")
mask = np.load(paths["mask"], mmap_mode="r")
cohort_user_ids = np.load(paths["cohort"]).astype(int)
movie_vocab = np.load(paths["vocab"]).astype(int)
user_means = np.load(paths["means"]).astype(np.float64)
personality = np.load(paths["personality"]).astype(np.float64)
mu_global = float(np.load(paths["mu_global"]))
test_labels = pd.read_csv(paths["test"])

n_users, n_movies, k_b1 = channelB1.shape
n_hidden = bh1.shape[0]
assert k_b1 == K
assert mask.shape == (n_users, n_movies)
assert W1.shape == (n_movies * K, n_hidden)
assert len(cohort_user_ids) == n_users
assert len(movie_vocab) == n_movies

user_to_row = {int(uid): i for i, uid in enumerate(cohort_user_ids)}
movie_to_col = {int(mid): j for j, mid in enumerate(movie_vocab)}

print(f"Project root: {root}")
print(f"n_users={n_users:,}  n_movies={n_movies:,}  n_hidden={n_hidden}")
print(f"W1 {W1.shape}  bh1 {bh1.shape}")
print(f"mu_global={mu_global:.6f}")

Project root: /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys
n_users=5,000  n_movies=13,129  n_hidden=128
W1 (131290, 128)  bh1 (128,)
mu_global=3.328083


In [2]:
# H1 = sigmoid(X1 @ W1 + bh1) — forward pass, reused from 14b_evaluation_5k.ipynb
X1 = np.asarray(channelB1, dtype=np.float32).reshape(n_users, -1)
H1 = sigmoid(X1 @ W1 + bh1)
print(f"H1 {H1.shape}")

# Decode train-only (user, movie, rating) triples directly from channelB1 + mask
# (already exactly the 80/20 temporal train split from 09b — no need to re-stream rating.csv)
print("Decoding train ratings from channelB1 one-hots …")
train_rows = []
step = 250
for i0 in range(0, n_users, step):
    i1 = min(i0 + step, n_users)
    m_block = np.asarray(mask[i0:i1])
    c_block = np.asarray(channelB1[i0:i1])
    for ii in range(i1 - i0):
        cols = np.where(m_block[ii] == 1)[0]
        if len(cols) == 0:
            continue
        ks = c_block[ii, cols, :].argmax(axis=1)
        ratings = RATING_LEVELS[ks]
        uid = int(cohort_user_ids[i0 + ii])
        mids = movie_vocab[cols]
        for mid, r in zip(mids, ratings):
            train_rows.append((uid, int(mid), float(r)))

train_df = pd.DataFrame(train_rows, columns=["userId", "movieId", "rating"])
print(f"train_df rows: {len(train_df):,}  (expected == mask.sum() == {int(np.asarray(mask).sum()):,})")
assert len(train_df) == int(np.asarray(mask).sum())

H1 (5000, 128)
Decoding train ratings from channelB1 one-hots …


train_df rows: 4,119,189  (expected == mask.sum() == 4,119,189)


## Part 1 — Movie-side metadata assembly

- `movie_meta`: title / genres / release year, restricted to `movie_vocab` (13,129 movies).
- Genome relevance matrix: MovieLens genome tags only cover 10,381 of the 27,279 movies overall (dense for those — 11,709,768 rows / 1,128 tags ≈ 10,381 exactly). Of `movie_vocab`, **10,239 / 13,129 (78%)** have genome coverage; the genome-tag evidence step below is restricted to that subset, not silently zero-filled for the rest.
- Per-movie free-text tag aggregation (top-5 most common), restricted to `movie_vocab`.

In [3]:
import re
from collections import Counter

vocab_set = set(int(m) for m in movie_vocab)

# 1a — title / genres / year, restricted to movie_vocab
movies_raw = pd.read_csv(data_dir / "movie.csv")
movies_raw = movies_raw[movies_raw["movieId"].isin(vocab_set)].copy()

YEAR_RE = re.compile(r"\((\d{4})\)\s*$")


def parse_year(title):
    m = YEAR_RE.search(str(title))
    return int(m.group(1)) if m else np.nan


movies_raw["year"] = movies_raw["title"].map(parse_year)
movies_raw["genre_list"] = movies_raw["genres"].fillna("(no genres listed)").map(
    lambda g: [] if g == "(no genres listed)" else g.split("|")
)
movie_meta = movies_raw.set_index("movieId")[["title", "genres", "genre_list", "year"]].to_dict("index")

n_with_year = sum(1 for j in movie_vocab if not np.isnan(movie_meta.get(int(j), {}).get("year", np.nan)))
print(f"movie_meta: {len(movie_meta):,} / {n_movies:,} vocab movies")
print(f"  year parsed for {n_with_year:,} movies")

all_genres_sorted = sorted({g for j in movie_vocab for g in movie_meta.get(int(j), {}).get("genre_list", [])})
print(f"  {len(all_genres_sorted)} distinct genres: {all_genres_sorted}")

corpus_genre_freq = Counter()
for j in movie_vocab:
    for g in movie_meta.get(int(j), {}).get("genre_list", []):
        corpus_genre_freq[g] += 1
corpus_mean_year = float(np.nanmean([movie_meta.get(int(j), {}).get("year", np.nan) for j in movie_vocab]))
print(f"  corpus mean year: {corpus_mean_year:.1f}")

movie_meta: 13,129 / 13,129 vocab movies
  year parsed for 13,124 movies
  19 distinct genres: ['Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'IMAX', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']
  corpus mean year: 1990.9


In [4]:
# 1b — genome relevance matrix, restricted to movie_vocab ∩ genome-covered movies
genome_tags_df = pd.read_csv(data_dir / "genome_tags.csv").sort_values("tagId")
tag_id_to_name = dict(zip(genome_tags_df["tagId"], genome_tags_df["tag"]))
genome_tag_ids = genome_tags_df["tagId"].to_numpy()
genome_tag_names = genome_tags_df["tag"].to_numpy()
n_tags = len(genome_tag_ids)

print("Loading genome_scores.csv …")
genome_scores_df = pd.read_csv(data_dir / "genome_scores.csv")
genome_scores_df = genome_scores_df[genome_scores_df["movieId"].isin(vocab_set)]

genome_movie_ids = np.sort(genome_scores_df["movieId"].unique())
print(f"genome coverage: {len(genome_movie_ids):,} / {n_movies:,} vocab movies")

genome_pivot = genome_scores_df.pivot(index="movieId", columns="tagId", values="relevance")
genome_pivot = genome_pivot.reindex(index=genome_movie_ids, columns=genome_tag_ids)
assert not genome_pivot.isna().any().any(), "Expected dense genome matrix for covered movies"
genome_matrix = genome_pivot.to_numpy(dtype=np.float64)  # (n_genome_movies, n_tags)

# Index positions of genome_movie_ids within movie_vocab order, for aligning with W1-derived vectors
genome_vocab_idx = np.array([movie_to_col[int(mid)] for mid in genome_movie_ids], dtype=np.int64)

print(f"genome_matrix {genome_matrix.shape}")
del genome_scores_df, genome_pivot

Loading genome_scores.csv …


genome coverage: 10,239 / 13,129 vocab movies


genome_matrix (10239, 1128)


In [5]:
# 1c — per-movie free-text tag aggregation (top-5 most common), restricted to movie_vocab
tags_raw = pd.read_csv(data_dir / "tag.csv", usecols=["movieId", "tag"])
tags_raw = tags_raw[tags_raw["movieId"].isin(vocab_set)]

movie_tags = {}
for mid, grp in tags_raw.groupby("movieId"):
    counts = Counter(t.strip().lower() for t in grp["tag"].dropna())
    movie_tags[int(mid)] = [t for t, _ in counts.most_common(5)]

n_with_tags = sum(1 for j in movie_vocab if int(j) in movie_tags)
print(f"movie_tags: {n_with_tags:,} / {n_movies:,} vocab movies have >=1 user tag")
del tags_raw

movie_tags: 12,061 / 13,129 vocab movies have >=1 user tag
